In [3]:
"""
main.py  —  AI-Powered Movie Recommendation System
============================================================
Topic 3 Hackathon Project
Combines:
  1. Content-Based Filtering (TF-IDF + cosine similarity)
  2. Collaborative Filtering (SciPy Pearson correlation)
  3. K-Means Clustering (SciPy cluster.vq)
  4. Visualizations (Matplotlib / Seaborn)
  5. Optional SciPy stats (chi-square, Spearman, t-test)
"""

import os, sys

os.chdir(r"C:\Users\DELL\Downloads\movie_recommender_project\movie_recommender")
sys.path.insert(0, os.getcwd())
import pandas as pd

# ── local modules ─────────────────────────────────────────
from recommender.content_based  import build_similarity_matrix, recommend_for_user
from recommender.collaborative  import build_rating_matrix, collaborative_recommend
from recommender.clustering     import cluster_users, cluster_recommendations
from recommender.scipy_stats    import test_genre_bias, spearman_similarity, compare_cluster_ratings
from visualizations.plots       import (plot_genre_pie, plot_ratings_over_time,
                                        plot_platform_genres, plot_cluster_heatmap)

# ── paths ─────────────────────────────────────────────────
BASE    = os.getcwd()
DATA    = os.path.join(BASE, "data")
OUT_DIR = os.path.join(BASE, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# ── 1. Load data ──────────────────────────────────────────
print("\n⏳  Loading datasets...")
movies_df  = pd.read_csv(os.path.join(DATA, "movies.csv"))
history_df = pd.read_csv(os.path.join(DATA, "viewing_history.csv"))
users_df   = pd.read_csv(os.path.join(DATA, "users.csv"))
print(f"  ✅  {len(movies_df)} movies | {len(users_df)} users | {len(history_df)} history records")

# ── 2. Build models ───────────────────────────────────────
print("\n⏳  Building recommendation models...")
cosine_sim    = build_similarity_matrix(movies_df)
rating_matrix = build_rating_matrix(history_df)
users_df      = cluster_users(users_df, history_df, k=4)
print("  ✅  Models ready.")

# ── 3. Demo: run for 3 sample users ───────────────────────
SAMPLE_USERS = ["U001", "U010", "U025"]

print("\n" + "═"*55)
print("   🎬  PERSONALIZED MOVIE RECOMMENDATIONS")
print("═"*55)

for uid in SAMPLE_USERS:
    row  = users_df[users_df["user_id"] == uid]
    name = row["name"].values[0] if not row.empty else uid

    # ── Content-based ────────────────────────────────────
    recommend_for_user(uid, users_df, movies_df, history_df, cosine_sim)

    # ── Collaborative filtering ──────────────────────────
    collaborative_recommend(uid, rating_matrix, history_df, users_df)

    # ── Cluster-based ────────────────────────────────────
    cluster_recommendations(uid, users_df, history_df)

# ── 4. Visualizations ────────────────────────────────────
print("\n⏳  Generating visualizations...")
TARGET = "U001"
trow   = users_df[users_df["user_id"] == TARGET].iloc[0]
tname  = trow["name"]

plot_genre_pie(TARGET, history_df, name=tname, output_dir=OUT_DIR)
plot_ratings_over_time(TARGET, history_df, name=tname, output_dir=OUT_DIR)
plot_platform_genres(history_df, output_dir=OUT_DIR)
plot_cluster_heatmap(users_df, history_df, output_dir=OUT_DIR)

# ── 5. Optional SciPy stats ───────────────────────────────
print("\n" + "═"*55)
print("   📐  OPTIONAL SCIPY STATISTICAL ANALYSIS")
print("═"*55)

test_genre_bias("U001", history_df)
spearman_similarity("U001", "U010", rating_matrix)
compare_cluster_ratings(0, 1, users_df, history_df)

print("\n" + "═"*55)
print("   ✅  All done! Visualizations saved to /outputs/")
print("═"*55 + "\n")



⏳  Loading datasets...
  ✅  30 movies | 100 users | 1054 history records

⏳  Building recommendation models...
  ✅  Models ready.

═══════════════════════════════════════════════════════
   🎬  PERSONALIZED MOVIE RECOMMENDATIONS
═══════════════════════════════════════════════════════

───────────────────────────────────────────────────────
  🎬  Content-Based Recs for Uma Brown (U001)
───────────────────────────────────────────────────────
  ✦  A Quiet Place                       [Horror]  ★ 9.5
  ✦  Joker                               [Drama]  ★ 9.4
  ✦  Midsommar                           [Horror]  ★ 8.9
  ✦  BlackKklansman                      [Drama]  ★ 8.1
  ✦  The Martian                         [Sci-Fi]  ★ 8.0


───────────────────────────────────────────────────────
  🤝  Collaborative Recs for Uma Brown (U001)
      Top similar users: ['U004', 'U013', 'U014']
───────────────────────────────────────────────────────
  ✦  The Social Network                  [Drama]  score: 25.60
  

c:\Users\DELL\Downloads\movie_recommender_project\movie_recommender\recommender\collaborative.py:34: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, p_val = pearsonr(target_vec[mask], other_vec[mask])
c:\Users\DELL\Downloads\movie_recommender_project\movie_recommender\recommender\collaborative.py:34: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, p_val = pearsonr(target_vec[mask], other_vec[mask])



───────────────────────────────────────────────────────
  🤝  Collaborative Recs for Ivan Jackson (U010)
      Top similar users: ['U002', 'U012', 'U014']
───────────────────────────────────────────────────────
  ✦  Avengers: Endgame                   [Action]  score: 24.30
  ✦  The Notebook                        [Romance]  score: 15.80
  ✦  The Social Network                  [Drama]  score: 14.00
  ✦  The Grand Budapest Hotel            [Comedy]  score: 12.90
  ✦  Spider-Man: Into the Spider-Verse   [Animation]  score: 12.80


───────────────────────────────────────────────────────
  📊  Cluster-Based Recs for Ivan Jackson (U010)
      Cluster #0  |  100 similar users
───────────────────────────────────────────────────────
  ✦  Inception                           [Sci-Fi]  cluster avg ★ 3.59
  ✦  Knives Out                          [Thriller]  cluster avg ★ 3.56
  ✦  Get Out                             [Horror]  cluster avg ★ 3.54
  ✦  Parasite                            [Thriller]  

c:\Users\DELL\anaconda3\Lib\site-packages\scipy\_lib\deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)
